# 🎨 NexEra - Professional Cartoon AI Personalizer (Stable Diffusion + IP-Adapter)
**Make sure GPU is enabled:** Runtime → Change runtime type → T4 GPU → Save

This notebook uses **Stable Diffusion (Ghibli Edition) + IP-Adapter** to natively redraw faces in perfect cartoon illustration style. No creepy cut-and-paste, no double eyes!

**NEW:** Includes Cell 6 to run Colab as an **AI Cloud Server** connected directly to your NexEra website!

In [ ]:
# ============================================================
# CELL 1: Install Professional AI Libraries & Server Tools
# ============================================================
!pip install -q diffusers transformers accelerate insightface onnxruntime-gpu pymupdf Pillow
!pip install -q fastapi uvicorn python-multipart pyngrok
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

import torch
print(f'✅ GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️  NO GPU! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ============================================================
# CELL 2: Load Stable Diffusion & IP-Adapter AI Models
# ============================================================
import torch
from diffusers import StableDiffusionImg2ImgPipeline
import insightface
from insightface.app import FaceAnalysis
import os

os.makedirs('/content/input', exist_ok=True)
os.makedirs('/content/output', exist_ok=True)

print('Loading Face Detector (CPU mode for maximum stability)...')
face_app = FaceAnalysis(name='buffalo_l', providers=['CPUExecutionProvider'])
face_app.prepare(ctx_id=-1, det_size=(640, 640))

face_app_small = FaceAnalysis(name='buffalo_l', providers=['CPUExecutionProvider'])
face_app_small.prepare(ctx_id=-1, det_size=(320, 320))

print('Loading Stable Diffusion (Ghibli Illustration Checkpoint)...')
pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    "nitrosocke/Ghibli-Diffusion",
    torch_dtype=torch.float16,
    safety_checker=None
).to("cuda")

print('Loading IP-Adapter (Face Identity Encoder)...')
pipe.load_ip_adapter("h94/IP-Adapter", subfolder="models", weight_name="ip-adapter-plus-face_sd15.bin")
pipe.set_ip_adapter_scale(0.7)  # 0.7 perfectly blends identity while keeping cartoon style

print('✅ Professional AI Generation Engine Ready!')

In [ ]:
# ============================================================
# CELL 3: Upload Child Photo + Story PDF (Manual Test)
# ============================================================
from google.colab import files
import glob

!rm -rf /content/input/*

print('📸 Upload the CHILD PHOTO (JPG/PNG) - clear front-facing photo:')
uploaded = files.upload()
child_photo_path = f"/content/input/{list(uploaded.keys())[0]}"
with open(child_photo_path, 'wb') as f:
    f.write(list(uploaded.values())[0])
print(f'✅ Child photo saved: {child_photo_path}')

print('\n📄 Upload the STORY PDF:')
uploaded2 = files.upload()
pdf_path = f"/content/input/{list(uploaded2.keys())[0]}"
with open(pdf_path, 'wb') as f:
    f.write(list(uploaded2.values())[0])
print(f'✅ PDF saved: {pdf_path}')

In [ ]:
# ============================================================
# CELL 4: Generate Masterpiece Storybook (Manual Test)
# ============================================================
import fitz, io, traceback
from PIL import Image, ImageDraw, ImageFilter
import cv2, numpy as np

def detect_face(img_cv):
    faces = face_app.get(img_cv)
    if faces:
        return sorted(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]), reverse=True)
    faces = face_app_small.get(img_cv)
    if faces:
        return sorted(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]), reverse=True)
    big = cv2.resize(img_cv, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    faces = face_app.get(big)
    if faces:
        scaled = []
        for f in faces:
            f.bbox = f.bbox / 2.0
            scaled.append(f)
        return sorted(scaled, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]), reverse=True)
    return []

def get_head_box(bbox, img_width, img_height, expand_ratio=1.8):
    x1, y1, x2, y2 = bbox
    w, h = x2 - x1, y2 - y1
    center_x, center_y = x1 + w / 2, y1 + h / 2
    size = max(w, h) * expand_ratio
    new_x1 = max(0, int(center_x - size / 2))
    new_y1 = max(0, int(center_y - size / 2))
    new_x2 = min(img_width, int(center_x + size / 2))
    new_y2 = min(img_height, int(center_y + size / 2))
    return (new_x1, new_y1, new_x2, new_y2)

def paste_with_feather(base_img, crop_img, box, feather=15):
    crop_resized = crop_img.resize((box[2]-box[0], box[3]-box[1]), Image.Resampling.LANCZOS)
    mask = Image.new("L", crop_resized.size, 0)
    draw = ImageDraw.Draw(mask)
    draw.rectangle([feather, feather, mask.width-feather, mask.height-feather], fill=255)
    mask = mask.filter(ImageFilter.GaussianBlur(feather//2))
    base_img.paste(crop_resized, (box[0], box[1]), mask)
    return base_img

def run_ai_pipeline(photo_path, pdf_path, output_pdf_path):
    print('Loading child photo for IP-Adapter...')
    child_pil = Image.open(photo_path).convert('RGB')
    child_pil_ip = child_pil.resize((512, 512), Image.Resampling.LANCZOS)
    print('✅ Child photo loaded!')

    doc = fitz.open(pdf_path)
    output_pdf = fitz.open()
    total = 0
    print(f'\n🎨 Redrawing {len(doc)} pages with Stable Diffusion...')

    prompt = "ghibli style cartoon illustration, cute child face, masterpiece, vibrant colors, high quality"
    negative_prompt = "ugly, blurry, bad anatomy, realistic, photorealistic, creepy, deformed, double eyes, extra limbs, poorly drawn face"
    generator = torch.Generator(device="cuda").manual_seed(42)

    for i in range(len(doc)):
        page = doc.load_page(i)
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
        page_pil = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        page_cv = cv2.cvtColor(np.array(page_pil), cv2.COLOR_RGB2BGR)
        
        target_faces = detect_face(page_cv)
        
        if target_faces:
            print(f'  Page {i+1}: Redrawing {len(target_faces)} character face(s)...')
            for tface in target_faces:
                box = get_head_box(tface.bbox, page_pil.width, page_pil.height, expand_ratio=1.8)
                if box[2]>box[0] and box[3]>box[1]:
                    head_crop = page_pil.crop(box).resize((512, 512), Image.Resampling.LANCZOS)
                    sd_result = pipe(
                        prompt=prompt,
                        negative_prompt=negative_prompt,
                        image=head_crop,
                        ip_adapter_image=child_pil_ip,
                        strength=0.55,
                        guidance_scale=7.5,
                        generator=generator,
                        num_inference_steps=25
                    ).images[0]
                    page_pil = paste_with_feather(page_pil, sd_result, box, feather=15)
                    total += 1
        else:
            print(f'  Page {i+1}: No characters detected, keeping original.')
        
        buf = io.BytesIO()
        page_pil.save(buf, format='JPEG', quality=95)
        img_doc = fitz.open('pdf', fitz.open(stream=buf.getvalue(), filetype='jpeg').convert_to_pdf())
        output_pdf.insert_pdf(img_doc)

    output_pdf.save(output_pdf_path)
    output_pdf.close()
    doc.close()
    print(f'\n✅ Masterpiece Complete! Redrew {total} faces.')
    return True

# Run manual test if files exist
if os.path.exists(child_photo_path) and os.path.exists(pdf_path):
    run_ai_pipeline(child_photo_path, pdf_path, '/content/output/personalized_story.pdf')
    print('Run Cell 5 to download manual test PDF, or run Cell 6 to start AI Cloud Server!')

In [ ]:
# ============================================================
# CELL 5: Download your Masterpiece Storybook (Manual Test)
# ============================================================
from google.colab import files
if os.path.exists('/content/output/personalized_story.pdf'):
    print('Downloading your professional personalized story...')
    files.download('/content/output/personalized_story.pdf')
    print('✅ Download started!')

In [ ]:
# ============================================================
# CELL 6: Start AI Cloud Server (Connect to NexEra Website)
# ============================================================
# Kill any existing server on port 5000 to prevent Address Already in Use errors
!fuser -k 5000/tcp || true

import subprocess
import time
import threading
import uvicorn
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import FileResponse
import shutil
import os

colab_app = FastAPI()

@colab_app.post("/colab_generate")
async def colab_generate(photo: UploadFile = File(...), pdf: UploadFile = File(...)):
    print(f"\n🚀 [COLAB SERVER] Received generation request from NexEra website!")
    print(f"  📸 Photo: {photo.filename}")
    print(f"  📄 PDF: {pdf.filename}")
    
    os.makedirs("/content/server_temp", exist_ok=True)
    photo_path = f"/content/server_temp/{photo.filename}"
    pdf_path = f"/content/server_temp/{pdf.filename}"
    out_path = f"/content/server_temp/final_masterpiece.pdf"
    
    with open(photo_path, "wb") as f:
        shutil.copyfileobj(photo.file, f)
    with open(pdf_path, "wb") as f:
        shutil.copyfileobj(pdf.file, f)
        
    print("⚙️ Running Stable Diffusion + IP-Adapter Masterpiece Pipeline...")
    try:
        success = run_ai_pipeline(photo_path, pdf_path, out_path)
        if success and os.path.exists(out_path):
            print("✅ [COLAB SERVER] Generation successful! Sending Masterpiece PDF back to website.")
            return FileResponse(out_path, media_type="application/pdf", filename="NexEra_Masterpiece.pdf")
    except Exception as e:
        print(f"❌ [COLAB SERVER] Generation failed: {e}")
        
    return {"error": "Colab AI generation failed"}

def run_server():
    uvicorn.run(colab_app, host="127.0.0.1", port=5000, log_level="warning")

# Start FastAPI in background thread
threading.Thread(target=run_server, daemon=True).start()
time.sleep(3)

# Start Cloudflare Tunnel
print("Opening secure Cloudflare HTTPS tunnel...")
process = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:5000'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(6)

# Find Cloudflare URL from stderr
url = None
for line in process.stderr:
    line = line.decode('utf-8')
    if "https://" in line and "trycloudflare.com" in line:
        url = [w for w in line.split() if "trycloudflare.com" in w][0]
        break

print("\n" + "="*75)
print("🌟 COLAB AI CLOUD SERVER IS LIVE AND READY! 🌟")
print(f"👉 COPY THIS URL: {url}")
print("👉 PASTE IT INTO THE NEXERA ADMIN DASHBOARD SETTINGS TAB!")
print("="*75 + "\n")